# Practical 1: CNN from Scratch (CIFAR-10)

## Objective
Build a Convolutional Neural Network (CNN) from scratch to classify images from the CIFAR-10 dataset.

## Dataset
CIFAR-10 consists of 60,000 32x32 color images in 10 classes:
- Airplane, Automobile, Bird, Cat, Deer, Dog, Frog, Horse, Ship, Truck
- 50,000 training images and 10,000 test images

## Step 1: Import Required Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")

## Step 2: Load and Explore the CIFAR-10 Dataset

In [ ]:
# Load CIFAR-10 dataset
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

print(f"Training data shape: {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"Test labels shape: {y_test.shape}")

# Class names
class_names = ['Airplane', 'Automobile', 'Bird', 'Cat', 'Deer', 
               'Dog', 'Frog', 'Horse', 'Ship', 'Truck']

## Step 3: Visualize Sample Images

In [ ]:
# Display sample images from each class
plt.figure(figsize=(15, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    # Find first image of this class
    idx = np.where(y_train == i)[0][0]
    plt.imshow(X_train[idx])
    plt.title(class_names[i])
    plt.axis('off')
plt.tight_layout()
plt.show()

## Step 4: Data Preprocessing

In [ ]:
# Normalize pixel values to [0, 1]
X_train_normalized = X_train.astype('float32') / 255.0
X_test_normalized = X_test.astype('float32') / 255.0

# Convert labels to one-hot encoding
y_train_categorical = to_categorical(y_train, 10)
y_test_categorical = to_categorical(y_test, 10)

print(f"Normalized training data range: [{X_train_normalized.min()}, {X_train_normalized.max()}]")
print(f"One-hot encoded label shape: {y_train_categorical.shape}")
print(f"Sample one-hot label: {y_train_categorical[0]}")

## Step 5: Build CNN Model from Scratch

### Architecture:
1. **Convolutional Layers**: Extract features from images
2. **Pooling Layers**: Reduce spatial dimensions
3. **Dropout Layers**: Prevent overfitting
4. **Dense Layers**: Classification

In [ ]:
def create_cnn_model():
    """
    Create a CNN model for CIFAR-10 classification
    """
    model = keras.Sequential([
        # First Convolutional Block
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', 
                     input_shape=(32, 32, 3), name='conv1'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', name='conv2'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2), name='pool1'),
        layers.Dropout(0.2),
        
        # Second Convolutional Block
        layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='conv3'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='conv4'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2), name='pool2'),
        layers.Dropout(0.3),
        
        # Third Convolutional Block
        layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='conv5'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='conv6'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2), name='pool3'),
        layers.Dropout(0.4),
        
        # Fully Connected Layers
        layers.Flatten(),
        layers.Dense(128, activation='relu', name='fc1'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax', name='output')
    ])
    
    return model

# Create the model
model = create_cnn_model()

# Display model architecture
model.summary()

## Step 6: Compile the Model

In [ ]:
# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully!")

## Step 7: Train the Model

In [ ]:
# Define callbacks
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7
)

# Train the model
history = model.fit(
    X_train_normalized, y_train_categorical,
    batch_size=64,
    epochs=50,
    validation_split=0.2,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

## Step 8: Visualize Training History

In [ ]:
# Plot training history
plt.figure(figsize=(12, 4))

# Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Step 9: Evaluate on Test Set

In [ ]:
# Evaluate on test data
test_loss, test_accuracy = model.evaluate(X_test_normalized, y_test_categorical, verbose=0)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

## Step 10: Make Predictions and Visualize Results

In [ ]:
# Make predictions
y_pred = model.predict(X_test_normalized)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = y_test.flatten()

# Display some predictions
plt.figure(figsize=(15, 8))
for i in range(20):
    plt.subplot(4, 5, i + 1)
    plt.imshow(X_test[i])
    true_label = class_names[y_true[i]]
    pred_label = class_names[y_pred_classes[i]]
    color = 'green' if y_true[i] == y_pred_classes[i] else 'red'
    plt.title(f"True: {true_label}\nPred: {pred_label}", color=color, fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()

## Step 11: Confusion Matrix

In [ ]:
# Create confusion matrix
cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## Step 12: Classification Report

In [ ]:
# Print classification report
print("\nClassification Report:")
print("=" * 70)
print(classification_report(y_true, y_pred_classes, target_names=class_names))

## Step 13: Save the Model (Optional)

In [ ]:
# Save the model
# model.save('cifar10_cnn_model.h5')
# print("Model saved successfully!")

## Summary

### What we learned:
1. **Loading CIFAR-10 dataset** from Keras datasets
2. **Data preprocessing**: Normalization and one-hot encoding
3. **Building a CNN from scratch** with:
   - Convolutional layers for feature extraction
   - Pooling layers for dimensionality reduction
   - Batch normalization for stable training
   - Dropout for regularization
4. **Training the model** with callbacks (early stopping, learning rate reduction)
5. **Evaluating performance** using accuracy, confusion matrix, and classification report
6. **Visualizing results** to understand model predictions

### Key Concepts:
- **Convolutional Layers**: Extract spatial features from images
- **Pooling Layers**: Reduce spatial dimensions while preserving important features
- **Batch Normalization**: Normalize activations to improve training stability
- **Dropout**: Randomly drop neurons during training to prevent overfitting
- **Softmax Activation**: Convert outputs to probability distribution for multi-class classification